# choosing files to be annotated

perhaps make a datasheet, all of the 5s clips, file paths, species detected (filtered), date, site, session (dawn/dusk), embedding parquet_path. in one .csv would be helpful i think.

Need to understand how the parquet embeddings work properly!

In [121]:
import pandas as pd
import polars as pl

# make panda display width 150 and max columns 20
pd.set_option('display.max_columns', 15)
pd.set_option('display.width', 300)
pl.Config.set_tbl_cols(15)
pl.Config.set_tbl_width_chars(300)

polars.config.Config

In [122]:
PT_ROOT = '/Users/ash/phd/wytham_pam_data/embeddings/2025'
PT_CHECKPOINT = f'{PT_ROOT}/checkpoint-10000'

In [123]:
emb = pd.read_parquet(PT_ROOT+'/G2-2/embeddings_partitioned/0/part-0.parquet', engine="pyarrow")
print(emb.shape)          # e.g. (14400, 1285)  — 14400 clips, 1280 emb cols + 5 metadata cols
print(emb.columns)    # file, file_path, chunk_idx, start_time, end_time, emb_0, ...

(800, 1541)
Index(['file', 'file_path', 'chunk_idx', 'start_time', 'end_time', 'emb_0', 'emb_1', 'emb_2', 'emb_3', 'emb_4',
       ...
       'emb_1526', 'emb_1527', 'emb_1528', 'emb_1529', 'emb_1530', 'emb_1531', 'emb_1532', 'emb_1533', 'emb_1534', 'emb_1535'], dtype='str', length=1541)


In [5]:
print(emb[(emb['file']=='20250714_194600_2')&(emb['chunk_idx']==2)])  # e.g. 0: 14400, 1: 14400, ... 9: 14400

                file                                          file_path  chunk_idx  start_time  end_time     emb_0     emb_1  ...  emb_1529  emb_1530  emb_1531  emb_1532  emb_1533  emb_1534  emb_1535
2  20250714_194600_2  /mnt/bio-lv-colefs01/2025-wytham-acoustics/G2-...          2        10.0      15.0 -0.044805 -0.092061  ... -0.070401 -0.114337  0.006316 -0.016656  0.045278 -0.030567 -0.128555

[1 rows x 1541 columns]


In [120]:

# Load the canonical index
index = pl.read_parquet("/Users/ash/Library/CloudStorage/OneDrive-Nexus365/PHD/Wytham_PAM/perch-cpu-inference/wytham_labels/output/canonical_clip_index.parquet")

# Query a specific clip
clip = index.filter(pl.col("clip_id") == "G2-2_20250714_194600_0")
print(clip)
# Get the embedding for this clip
emb_path = clip["embedding_path"][0]
emb_df = pl.read_parquet(emb_path)
clip_embedding = emb_df.filter(
    (pl.col("chunk_idx") == 0) & 
    (pl.col("file_path").str.contains("20250714_194600"))
)

print(clip_embedding.select(pl.col("^emb_.*$")))  # All 1280 embedding dimensions

shape: (1, 24)
┌────────────────────────┬──────┬──────────────────────────────────┬─────────────────────┬───────────┬────────────┬──────────┬──────────────┬───┬──────────────┬────────────┬──────────────────────────────────┬───────────────────────┬──────────────────────────────────┬──────────────────┬─────────────┐
│ clip_id                ┆ site ┆ audio_path                       ┆ file_name           ┆ chunk_idx ┆ start_time ┆ end_time ┆ duration_sec ┆ … ┆ top1_species ┆ top1_score ┆ top_species                      ┆ top_scores            ┆ embedding_path                   ┆ embedding_exists ┆ file_exists │
│ ---                    ┆ ---  ┆ ---                              ┆ ---                 ┆ ---       ┆ ---        ┆ ---      ┆ ---          ┆   ┆ ---          ┆ ---        ┆ ---                              ┆ ---                   ┆ ---                              ┆ ---              ┆ ---         │
│ str                    ┆ str  ┆ str                              ┆ str          

In [7]:
missing_embeddings = index.filter(pl.col("embedding_exists") == False)


missing_embeddings.head()

clip_id,site,audio_path,file_name,chunk_idx,start_time,end_time,duration_sec,datetime_start,datetime_end,year,month,day,hour,minute,date,checkpoint_id,top1_species,top1_score,top_species,top_scores,embedding_path,embedding_exists,file_exists
str,str,str,str,i64,f64,f64,f64,datetime[μs],datetime[μs],i32,i8,i8,i8,i8,date,i64,str,f64,list[str],list[f64],str,bool,bool
"""H9_20250216_063100_7""","""H9""","""/mnt/bio-lv-colefs01/2025-wyth…","""20250216_063100.WAV""",7,35.0,40.0,5.0,2025-02-16 06:31:35,2025-02-16 06:31:40,2025,2,16,6,31,2025-02-16,1,"""Strix aluco""",7.570748,"[""Strix aluco"", ""Turdus pilaris"", … ""Turdus merula""]","[7.570748, 7.329322, … 6.029548]","""/Users/ash/phd/wytham_pam_data…",false,false
"""H9_20250216_063100_8""","""H9""","""/mnt/bio-lv-colefs01/2025-wyth…","""20250216_063100.WAV""",8,40.0,45.0,5.0,2025-02-16 06:31:40,2025-02-16 06:31:45,2025,2,16,6,31,2025-02-16,1,"""Strix aluco""",8.078139,"[""Strix aluco"", ""Prunella collaris"", … ""Passer domesticus""]","[8.078139, 6.869777, … 6.076289]","""/Users/ash/phd/wytham_pam_data…",false,false
"""H9_20250216_063100_9""","""H9""","""/mnt/bio-lv-colefs01/2025-wyth…","""20250216_063100.WAV""",9,45.0,50.0,5.0,2025-02-16 06:31:45,2025-02-16 06:31:50,2025,2,16,6,31,2025-02-16,1,"""Strix aluco""",7.76244,"[""Strix aluco"", ""Asio otus"", … ""Turdus merula""]","[7.76244, 6.471144, … 5.946932]","""/Users/ash/phd/wytham_pam_data…",false,false
"""H9_20250216_063100_10""","""H9""","""/mnt/bio-lv-colefs01/2025-wyth…","""20250216_063100.WAV""",10,50.0,55.0,5.0,2025-02-16 06:31:50,2025-02-16 06:31:55,2025,2,16,6,31,2025-02-16,1,"""Strix aluco""",8.270762,"[""Strix aluco"", ""Bubo virginianus"", … ""Gallinula chloropus""]","[8.270762, 8.065734, … 5.747013]","""/Users/ash/phd/wytham_pam_data…",false,false
"""H9_20250216_063100_11""","""H9""","""/mnt/bio-lv-colefs01/2025-wyth…","""20250216_063100.WAV""",11,55.0,60.0,5.0,2025-02-16 06:31:55,2025-02-16 06:32:00,2025,2,16,6,31,2025-02-16,1,"""Strix aluco""",8.153281,"[""Strix aluco"", ""Bubo bubo"", … ""Sylvia atricapilla""]","[8.153281, 7.200423, … 5.99047]","""/Users/ash/phd/wytham_pam_data…",false,false


In [104]:
# list of 5 missing embeddings paths
missing_embeddings_paths = missing_embeddings[["embedding_path",'checkpoint_id']].unique().head(20)
missing_embeddings[["embedding_path",'checkpoint_id']].unique().head(20)


embedding_path,checkpoint_id
str,i64
"""/Users/ash/phd/wytham_pam_data…",71
"""/Users/ash/phd/wytham_pam_data…",95
"""/Users/ash/phd/wytham_pam_data…",48
"""/Users/ash/phd/wytham_pam_data…",120
"""/Users/ash/phd/wytham_pam_data…",112
…,…
"""/Users/ash/phd/wytham_pam_data…",49
"""/Users/ash/phd/wytham_pam_data…",98
"""/Users/ash/phd/wytham_pam_data…",136


In [116]:
# get embedding for a given record
clip_id = "G2-2_20250714_194600_1"
clip = index.filter(pl.col("clip_id") == clip_id)
print(clip)
emb_path = clip["embedding_path"][0]
emb_df = pl.read_parquet(emb_path)
clip_embedding = emb_df.filter(
    (pl.col("chunk_idx") == clip['chunk_idx'][0]) & 
    (pl.col("file_path").str.contains(clip_id.split('_')[1] + '_' + clip_id.split('_')[2]))  # match date and time, turns G2-2_20250714_194600_0 into 20250714_194600. (splits on _ and picks out the date and time parts)
)
print(clip_embedding)

shape: (1, 24)
┌────────────────────────┬──────┬──────────────────────────────────┬─────────────────────┬───────────┬────────────┬──────────┬──────────────┬───┬──────────────────────┬────────────┬──────────────────────────┬───────────────────────┬──────────────────────────────────┬──────────────────┬─────────────┐
│ clip_id                ┆ site ┆ audio_path                       ┆ file_name           ┆ chunk_idx ┆ start_time ┆ end_time ┆ duration_sec ┆ … ┆ top1_species         ┆ top1_score ┆ top_species              ┆ top_scores            ┆ embedding_path                   ┆ embedding_exists ┆ file_exists │
│ ---                    ┆ ---  ┆ ---                              ┆ ---                 ┆ ---       ┆ ---        ┆ ---      ┆ ---          ┆   ┆ ---                  ┆ ---        ┆ ---                      ┆ ---                   ┆ ---                              ┆ ---              ┆ ---         │
│ str                    ┆ str  ┆ str                              ┆ str          